# CARD-1 `avg_frozen` ×3 seeds — Colab runner

Trains the **frozen-q ablation** (= AVG in-house, kills the recipe confound) on Colab GPU,
then evals each seed with `eval_union` (APGD, n=1000) under the locked protocol
(ε∞=0.03, ℓ2=0.5, ℓ1=12).

**Before running:** Runtime → Change runtime type → GPU (T4 ok, A100 faster).
Repo must be pushed to GitHub first (private → add a fine-grained token below).

In [ ]:
# 1) Clone (private repo: paste a GitHub fine-grained PAT when prompted)
import os, getpass
if not os.path.exists('attackdro'):
    tok = getpass.getpass('GitHub token (blank if repo public): ')
    url = f'https://{tok}@github.com/anhkiet287/attackdro.git' if tok else 'https://github.com/anhkiet287/attackdro.git'
    !git clone -q {url} attackdro
%cd attackdro
!pip -q install -r requirements.txt
!nvidia-smi -L

In [ ]:
# 2) CIFAR-10 (torchvision downloads; if the toronto mirror is slow, retry)
import torchvision; torchvision.datasets.CIFAR10('data/', download=True)

In [ ]:
# 3) Smoke first (2 steps) — do NOT skip
!python scripts/train.py --config configs/avg_frozen.yaml --smoke --wandb-mode disabled

In [ ]:
# 4) Train + eval 3 seeds sequentially (~2h/seed on T4; wandb offline by default here)
import subprocess
for S in [0, 1, 2]:
    run = f'avg_frozen_s{S}'
    print(f'=== TRAIN {run} ===', flush=True)
    subprocess.run(['python', 'scripts/train.py', '--config', 'configs/avg_frozen.yaml',
                    '--seed', str(S), '--run-name', run, '--wandb-mode', 'offline'], check=True)
    print(f'=== EVAL {run} ===', flush=True)
    subprocess.run(['python', 'scripts/evaluate.py', '--config', 'configs/avg_frozen.yaml',
                    '--checkpoint', f'checkpoints/{run}_best.pt', '-n', '1000',
                    '--version', 'apgd', '--out', f'results/eval_{run}.json'], check=True)

In [ ]:
# 5) Table + copy results to Drive (so the PC repo can ingest the JSONs)
import json, glob
print('| run | clean | l_inf | l2 | l1 | worst-union |')
print('|---|---|---|---|---|---|')
for p in sorted(glob.glob('results/eval_avg_frozen_s*.json')):
    m = json.load(open(p))['metrics']
    r = m['per_norm_robust_acc']
    print(f"| {p.split('/')[-1]} | {m['clean_acc']*100:.1f} | {r['linf']*100:.1f} | "
          f"{r['l2']*100:.1f} | {r['l1']*100:.1f} | **{m['worst_union_acc']*100:.1f}** |")
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/attackdro_results && cp results/eval_avg_frozen_s*.json checkpoints/avg_frozen_s*_best.pt /content/drive/MyDrive/attackdro_results/
print('copied to Drive: attackdro_results/')